# CS229 L07 — Neural Networks: Architecture

**Video:** Spring 2026 · [YouTube](https://www.youtube.com/watch?v=fRM41w9jzQo)  
**Instructor:** Tengyu Ma  
**Topics:** Nonlinear models · Loss functions · SGD · ReLU · Neurons · Layers · MLP · Residual connections · Layer Norm / RMSNorm · Convolutions

---

> 📌 *Lecture:* "These two lectures are mostly just setting up the basics of deep learning, especially in the context of supervised learning. We're going to introduce a general framework for nonlinear models — pretty much the same as what you have seen in linear models, just extended in a simple way."

---

## Roadmap

| Section | Key Idea |
|---|---|
| 1. Framework for nonlinear models | Same as linear, but $h_\theta$ is now nonlinear |
| 2. Loss functions | MSE (regression) · Cross-entropy (classification) |
| 3. SGD and mini-batch SGD | Gradient is unbiased estimator; why not full gradient |
| 4. ReLU and the single neuron | First nonlinear primitive |
| 5. Multi-layer networks (MLP) | Stack neurons → layers → deep network |
| 6. Other activation functions | Sigmoid · tanh · Leaky ReLU · GELU |
| 7. Residual connections | Learn the difference, not the function |
| 8. Layer Norm and RMSNorm | Prevent activation explosion; scale invariance |
| 9. Convolutional networks | Parameter sharing via Toeplitz matrices |


## 1. Framework for Nonlinear Models

### Extending linear models

**Linear model:** $h_\theta(x) = \theta^T x = \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_d x_d$

**Nonlinear model:** $h_\theta(x)$ can be any function nonlinear in $\theta$.

**Important distinction:**
- *Nonlinear in $x$ only* (e.g., $\theta_1 x_1^2 + \theta_2 x_2^2$): substitute $z_i = x_i^2$ → still a linear model in $z$. No new learning machinery needed.
- *Nonlinear in $\theta$* (e.g., $\theta_1^2 x_1$): cannot be reduced to linear model. Needs different techniques.

Neural networks are nonlinear in $\theta$. This is what makes them interesting and hard.

### The supervised learning framework (same as before)

Given dataset $\{(x^{(i)}, y^{(i)})\}_{i=1}^n$ with $x^{(i)} \in \mathbb{R}^d$:

1. **Choose a model class** $h_\theta$ (now a neural network)
2. **Define a loss** $J_i(\theta) = \text{Loss}(h_\theta(x^{(i)}), y^{(i)})$
3. **Minimize average loss** $J(\theta) = \frac{1}{n} \sum_{i=1}^n J_i(\theta)$ over $\theta$

Two remaining questions for this lecture:
- **Q1:** How do we define $h_\theta$? (→ Neural network architecture)
- **Q2:** How do we compute $\nabla_\theta J_i$? (→ Backpropagation, L08)


## 2. Loss Functions

### Regression: Mean Squared Error

For $y^{(i)} \in \mathbb{R}$:

$$
J_i(\theta) = \left(y^{(i)} - h_\theta(x^{(i)})\right)^2
$$

**MLE derivation:** assume $y = h_\theta(x) + \varepsilon$, $\varepsilon \sim \mathcal{N}(0, \sigma^2)$. Then:

$$
\log p(y \mid x) = \text{const} - \frac{1}{2\sigma^2}(y - h_\theta(x))^2
$$

Maximizing log-likelihood = minimizing MSE. Same result as L03, now with any $h_\theta$.

### Classification: Cross-Entropy Loss

For $K$-class classification with $y^{(i)} \in \{1, \ldots, K\}$:

**Model:** $\bar{h}_\theta: \mathbb{R}^d \to \mathbb{R}^K$ outputs **logits** (unnormalized scores).

**Predicted probability** of class $j$:

$$
p_\theta(y = j \mid x) = \frac{e^{\bar{h}_\theta(x)_j}}{\sum_{k=1}^K e^{\bar{h}_\theta(x)_k}} = \text{softmax}(\bar{h}_\theta(x))_j
$$

**Cross-entropy loss** for example $(x, y)$:

$$
J_i(\theta) = -\log p_\theta(y^{(i)} \mid x^{(i)}) = -\log \frac{e^{\bar{h}_\theta(x^{(i)})_{y^{(i)}}}}{\sum_{k=1}^K e^{\bar{h}_\theta(x^{(i)})_k}}
$$

$$
= -\bar{h}_\theta(x^{(i)})_{y^{(i)}} + \log \sum_{k=1}^K e^{\bar{h}_\theta(x^{(i)})_k}
$$

> 📌 *Lecture:* "If you use PyTorch to implement it, I think you can find a module which is just called `CrossEntropyLoss` and you give it two inputs and you can compute it — but of course the underlying math is this."

**Why cross-entropy?** It equals the cross-entropy between the predicted distribution $p_\theta(\cdot|x)$ and the true label distribution $p^*(\cdot|x)$ (in expectation over $(x,y)$ from the data distribution).

---

> 🎯 **Interview:** What is cross-entropy loss and how does it relate to MLE?
> 
> **A:** Cross-entropy loss $-\log p_\theta(y|x)$ is the negative log-likelihood of the true label under the model's predicted distribution. Minimizing cross-entropy = maximizing log-likelihood = MLE. For binary classification this reduces to binary cross-entropy $-(y\log\hat{p} + (1-y)\log(1-\hat{p}))$, which we derived from the Bernoulli MLE in L03. For multiclass it generalizes via softmax + log.


## 3. SGD and Mini-Batch SGD

### Why not full gradient descent?

Full gradient: $\nabla J(\theta) = \frac{1}{n} \sum_{i=1}^n \nabla J_i(\theta)$

**Problem:** $n$ is enormous. In 2015: ~1M examples. In 2026: ~1 trillion tokens. Computing all $n$ gradients per step is infeasible.

> 📌 *Lecture:* "Computing one gradient is very costly let alone computing all the gradients, so that's why you have to find an efficient way."

### SGD (batch size 1)

Sample a single example $j \sim \text{Uniform}\{1, \ldots, n\}$, update:

$$
\theta \leftarrow \theta - \eta \, \nabla J_j(\theta)
$$

**Why it works:** the single-example gradient is an **unbiased estimator** of the full gradient:

$$
\mathbb{E}_{j}\!\left[\nabla J_j(\theta)\right] = \frac{1}{n} \sum_{i=1}^n \nabla J_i(\theta) = \nabla J(\theta)
$$

In expectation, SGD moves in the right direction. It's just noisy.

### Mini-batch SGD (practical default)

Sample a **batch** $\mathcal{B}$ of $B$ examples:

$$
\theta \leftarrow \theta - \eta \cdot \frac{1}{B} \sum_{j \in \mathcal{B}} \nabla J_j(\theta)
$$

**Tradeoffs:**

| Batch size | Pros | Cons |
|---|---|---|
| $B=1$ | Fastest update | Very noisy gradient; poor GPU utilization |
| $B=n$ | Exact gradient | Too slow; can't fit in memory |
| $B \in [32, 4096]$ | Good GPU utilization | Some noise; tunable tradeoff |

### Do local minima hurt?

In 1D: many local minima are easy to get stuck in.

In high dimensions ($d$ large): a local minimum requires the Hessian to be **positive semi-definite** at a gradient-zero point — a very hard constraint to satisfy simultaneously in all $d$ directions.

> 📌 *Lecture:* "In high dimensional cases, it's not that trivial to have that many local minima. The common hypothesized consensus is that there are very limited numbers of bad local minima in this high dimensional function. Maybe there's even no bad local minimum — all the local minima are actually global minima."

---

> 🎯 **Interview:** Why does SGD work despite using only one example per gradient step?
> 
> **A:** The single-example gradient $\nabla J_j(\theta)$ is an unbiased estimator of the full gradient $\nabla J(\theta)$: $\mathbb{E}_j[\nabla J_j] = \nabla J$. So in expectation, every SGD step moves in the correct direction. Individual steps are noisy but the noise averages out over many steps. This allows training on trillion-token datasets where full gradients would be computationally impossible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Visualize SGD vs full GD on a simple quadratic
# True loss: J(theta) = 0.5 * theta^2 (minimum at 0)
# Per-example loss: J_i(theta) = 0.5*(theta - noise_i)^2

n = 100
noise = np.random.randn(n) * 2   # per-example offsets
# Full gradient: dJ/dtheta = theta
# Per-example gradient: dJ_i/dtheta = theta - noise_i

def full_grad(theta):
    return theta  # exact

def sgd_grad(theta):
    j = np.random.randint(n)
    return theta - noise[j]  # noisy

eta = 0.1
n_steps = 60
theta0 = 5.0

# Full GD
theta_gd = theta0
path_gd = [theta_gd]
for _ in range(n_steps):
    theta_gd -= eta * full_grad(theta_gd)
    path_gd.append(theta_gd)

# SGD
theta_sgd = theta0
path_sgd = [theta_sgd]
for _ in range(n_steps):
    theta_sgd -= eta * sgd_grad(theta_sgd)
    path_sgd.append(theta_sgd)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trajectories
ax = axes[0]
ax.plot(path_gd, 'b-o', ms=4, label='Full GD (exact)', linewidth=2)
ax.plot(path_sgd, 'r-s', ms=4, alpha=0.7, label='SGD (noisy)', linewidth=1.5)
ax.axhline(0, color='k', linestyle='--', linewidth=1, label='Minimum θ=0')
ax.set_xlabel('Step'); ax.set_ylabel('θ')
ax.set_title('GD vs SGD: Parameter Trajectory')
ax.legend(); ax.grid(True, alpha=0.3)

# Loss curves
ax = axes[1]
ax.plot([0.5*t**2 for t in path_gd], 'b-', label='Full GD loss', linewidth=2)
ax.plot([0.5*t**2 for t in path_sgd], 'r-', alpha=0.7, label='SGD loss', linewidth=1.5)
ax.set_xlabel('Step'); ax.set_ylabel('J(θ) = 0.5θ²')
ax.set_title('Loss vs Step')
ax.set_yscale('log'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('SGD converges in expectation; individual steps are noisy', fontsize=12)
plt.tight_layout()
plt.show()

print("GD: smooth convergence to minimum")
print("SGD: noisy but converges — in expectation, each step is correct")

## 4. ReLU and the Single Neuron

### Motivation: housing price

Real housing price vs. size might look like: flat at first, then roughly linear. A linear model can't capture this shape. We need a curve:

$$
\hat{y} = \max(wx + b, 0) + c
$$

This cuts off the negative part (prices can't be negative) and shifts vertically.

### ReLU: Rectified Linear Unit

$$
\text{ReLU}(t) = \max(t, 0) = t^+ = \begin{cases} t & t > 0 \\ 0 & t \leq 0 \end{cases}
$$

This is the most important activation function in modern deep learning.

**Properties:**
- Linear on the positive half, zero on the negative half
- Gradient = 1 for $t > 0$, 0 for $t < 0$ (undefined at 0, set to 0 by convention)
- Applied **elementwise** to vectors: $\text{ReLU}(v)_i = \max(v_i, 0)$

**Biological motivation:** neurons "fire" (activate) when input exceeds a threshold; silent below it. ReLU approximates this.

### Single neuron: high-dimensional input

For $x \in \mathbb{R}^d$, a single neuron computes:

$$
a = \text{ReLU}(w^T x + b)
$$

where $w \in \mathbb{R}^d$ is the **weight vector** and $b \in \mathbb{R}$ is the **bias**. Output is a scalar.

Parameters: $d + 1$ (weights + bias).

> 📌 *Lecture:* "We have already defined a one-dimensional network. This is a neural network with one-dimensional input and a few parameters. That's already a neural network."

---

> 🎯 **Interview:** Why do we need activation functions in neural networks?
> 
> **A:** Without an activation function, stacking linear layers composes to another linear function: $W_2(W_1 x + b_1) + b_2 = (W_2 W_1)x + (W_2 b_1 + b_2)$. No matter how many layers, the whole network is still linear. Activation functions like ReLU break this linearity, allowing the network to represent arbitrary nonlinear functions. This is what gives neural networks their expressive power.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(-3, 3, 300)

# Activation functions
relu = np.maximum(t, 0)
sigmoid = 1 / (1 + np.exp(-t))
tanh = np.tanh(t)
leaky_relu = np.where(t >= 0, t, 0.1 * t)
# GELU approximation
gelu = 0.5 * t * (1 + np.tanh(np.sqrt(2/np.pi) * (t + 0.044715 * t**3)))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
configs = [
    ('ReLU', relu, 'C0'),
    ('Sigmoid', sigmoid, 'C1'),
    ('Tanh', tanh, 'C2'),
    ('Leaky ReLU (α=0.1)', leaky_relu, 'C3'),
    ('GELU', gelu, 'C4'),
]
for ax, (name, y, color) in zip(axes, configs):
    ax.plot(t, y, color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('t'); ax.set_ylim(-1.5, 2.5)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('σ(t)')
plt.suptitle('Activation Functions', fontsize=13)
plt.tight_layout()
plt.show()

print("ReLU: default choice. Simple, fast, no vanishing gradient for t>0.")
print("Sigmoid: used in output layer for binary classification. Saturates → vanishing gradient.")
print("GELU: smooth approximation of ReLU, used in GPT/BERT.")
print("Leaky ReLU: prevents dead neurons (gradient=0.1 for t<0 instead of 0).")

## 5. Multi-Layer Networks (MLP)

### From one neuron to one layer

Stack $m$ neurons, each looking at all of $x \in \mathbb{R}^d$:

$$
a_1 = \text{ReLU}(w_1^T x + b_1), \quad a_2 = \text{ReLU}(w_2^T x + b_2), \quad \ldots, \quad a_m = \text{ReLU}(w_m^T x + b_m)
$$

In matrix form, collecting all $w_j$ as rows of $W^{[1]} \in \mathbb{R}^{m \times d}$:

$$
a = \text{ReLU}(W^{[1]} x + b^{[1]})
$$

where $a \in \mathbb{R}^m$ (the hidden layer), $b^{[1]} \in \mathbb{R}^m$.

**Parameters in layer 1:** $m \times d + m = m(d+1)$

### Two-layer network

Add a second (output) layer on top of $a$:

$$
h_\theta(x) = W^{[2]} a + b^{[2]} = W^{[2]}\, \text{ReLU}(W^{[1]} x + b^{[1]}) + b^{[2]}
$$

where $W^{[2]} \in \mathbb{R}^{k \times m}$ (for $k$ outputs), $b^{[2]} \in \mathbb{R}^k$.

**Motivation (housing price):** $a_1$ might capture "walkability", $a_2$ "school quality" — intermediate features not directly in the data, computed as nonlinear functions of raw inputs.

### Deep network: $L$ layers

Apply $\sigma(Wz + b)$ repeatedly:

$$
z^{[1]} = \sigma(W^{[1]} x + b^{[1]})
$$
$$
z^{[2]} = \sigma(W^{[2]} z^{[1]} + b^{[2]})
$$
$$
\vdots
$$
$$
h_\theta(x) = W^{[L]} z^{[L-1]} + b^{[L]}
$$

Each $\sigma(Wz + b)$ is one **layer**. The number of layers = $L$.

### Dimension compatibility

If layer $\ell$ outputs dimension $m_\ell$, then $W^{[\ell+1]}$ must have $m_\ell$ columns. Dimensions must chain:

```
x: (d,) → W^[1]: (m1, d) → z^[1]: (m1,) → W^[2]: (m2, m1) → z^[2]: (m2,) → ...
```

### Parameter counting

For a network with dimensions $d, m_1, m_2, \ldots, m_{L-1}, k$:

$$
\text{Total params} = \sum_{\ell=1}^L (m_{\ell} \times m_{\ell-1} + m_{\ell})
$$

where $m_0 = d$ (input) and $m_L = k$ (output).

> 📌 *Lecture:* "The input dimension is fixed by the data. You shrink it to some dimension and then keep using that dimension. For language, initial dimension could be like 250K (vocab size), you shrink to 2K and keep using 2K."

### MLP = Multi-Layer Perceptron

The term **MLP** usually refers to 1–3 layers of (linear + activation) blocks. In transformer architectures, "MLP" typically means 2 linear layers with one activation in between.

---

> 🎯 **Interview:** Why can't we just stack linear layers without activation functions?
> 
> **A:** Composition of linear functions is linear: $W_2(W_1 x + b_1) + b_2 = (W_2W_1)x + (W_2b_1+b_2)$. The whole stack collapses to a single linear layer regardless of depth. Activation functions like ReLU introduce nonlinearity, breaking this collapse. By the universal approximation theorem, a two-layer network with enough hidden units can approximate any continuous function — but only because of the activation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# --- Build a simple 2-layer MLP from scratch ---

class MLP:
    """2-layer MLP: input_dim → hidden_dim → output_dim."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        # Xavier initialization
        scale1 = np.sqrt(2.0 / input_dim)
        scale2 = np.sqrt(2.0 / hidden_dim)
        self.W1 = np.random.randn(hidden_dim, input_dim) * scale1  # (m, d)
        self.b1 = np.zeros(hidden_dim)                              # (m,)
        self.W2 = np.random.randn(output_dim, hidden_dim) * scale2  # (k, m)
        self.b2 = np.zeros(output_dim)                              # (k,)

    def forward(self, x):
        # Layer 1: linear + ReLU
        z1 = x @ self.W1.T + self.b1   # (n, m)
        a1 = np.maximum(z1, 0)          # ReLU, (n, m)
        # Layer 2: linear (no activation for regression output)
        z2 = a1 @ self.W2.T + self.b2   # (n, k)
        return z2

# Approximate sin(x) with an MLP via SGD
def train_mlp(hidden_dim=64, lr=0.01, n_steps=2000, batch_size=32):
    mlp = MLP(input_dim=1, hidden_dim=hidden_dim, output_dim=1)
    x_train = np.random.uniform(-np.pi, np.pi, 500).reshape(-1, 1)
    y_train = np.sin(x_train).reshape(-1, 1)

    losses = []
    for step in range(n_steps):
        idx = np.random.choice(len(x_train), batch_size)
        xb, yb = x_train[idx], y_train[idx]

        # Forward
        z1 = xb @ mlp.W1.T + mlp.b1
        a1 = np.maximum(z1, 0)
        z2 = a1 @ mlp.W2.T + mlp.b2
        pred = z2

        # Loss (MSE)
        diff = pred - yb
        loss = (diff**2).mean()
        losses.append(loss)

        # Backprop (manual for now — full derivation in L08)
        dz2 = 2 * diff / batch_size             # (B, 1)
        dW2 = dz2.T @ a1                        # (1, m)
        db2 = dz2.sum(axis=0)                   # (1,)
        da1 = dz2 @ mlp.W2                      # (B, m)
        dz1 = da1 * (z1 > 0)                    # ReLU gradient
        dW1 = dz1.T @ xb                        # (m, 1)
        db1 = dz1.sum(axis=0)

        mlp.W2 -= lr * dW2
        mlp.b2 -= lr * db2
        mlp.W1 -= lr * dW1
        mlp.b1 -= lr * db1

    return mlp, losses

mlp, losses = train_mlp(hidden_dim=64, lr=0.01, n_steps=3000)

x_test = np.linspace(-np.pi, np.pi, 200).reshape(-1, 1)
y_pred = mlp.forward(x_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_test, np.sin(x_test), 'k--', linewidth=2, label='sin(x) (target)')
axes[0].plot(x_test, y_pred, 'C0', linewidth=2, label='MLP prediction')
axes[0].set_title('2-Layer MLP Approximating sin(x)', fontsize=12)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(losses, 'C1', linewidth=1, alpha=0.7)
axes[1].set_title('Training Loss (MSE)', fontsize=12)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)

plt.suptitle('MLP from scratch: linear → ReLU → linear', fontsize=12)
plt.tight_layout()
plt.show()

n_params = mlp.W1.size + mlp.b1.size + mlp.W2.size + mlp.b2.size
print(f"Architecture: 1 → 64 → 1")
print(f"Parameters: W1({mlp.W1.shape}) + b1({mlp.b1.shape}) + W2({mlp.W2.shape}) + b2({mlp.b2.shape}) = {n_params} total")

## 6. Other Activation Functions

All activations are scalar functions $\mathbb{R} \to \mathbb{R}$, applied elementwise to vectors.

| Activation | Formula | Key property |
|---|---|---|
| **ReLU** | $\max(t, 0)$ | Default; fast; gradient dies for $t<0$ |
| **Sigmoid** | $\frac{1}{1+e^{-t}}$ | Output in $(0,1)$; saturates → vanishing gradient |
| **Tanh** | $\frac{e^t - e^{-t}}{e^t + e^{-t}}$ | Zero-centered; saturates at $\pm 1$ |
| **Leaky ReLU** | $\max(t, \alpha t),\ \alpha \approx 0.01$ | Fixes "dead ReLU" (small gradient for $t<0$) |
| **GELU** | $t \cdot \Phi(t)$ | Smooth; slight dip below zero; used in GPT/BERT |
| **SiLU/Swish** | $t \cdot \sigma(t)$ | Self-gated; used in LLaMA, PaLM |

### Why GELU over ReLU in modern LLMs?

GELU is smooth everywhere (differentiable at 0). Smooth activations tend to produce better-conditioned optimization landscapes, especially in very deep networks. ReLU's kink at 0 creates abrupt gradient changes.

> 📌 *Lecture:* "People seem to find that using somewhat smooth functions is actually useful. ReLU is actually not necessarily great because it's not smooth enough. Other than that I don't really know exactly — there's like a stability thing, sometimes just trial and error."

**Dead ReLU problem:** if $w^Tx + b < 0$ for all training examples, the neuron outputs 0 and its gradient is 0. The weights never update. That neuron is "dead". Leaky ReLU/GELU prevent this.


## 7. Residual Connections

### The motivation

Suppose at some intermediate layer, $z$ is already close to the target $y$. Then you only need to learn the **residual** (difference) $y - z$, not the full function from scratch.

If you believe: $y - z \approx f(z)$ for some network $f$, then:

$$y = f(z) + z$$

So instead of parameterizing the full mapping from $z$ to $y$, you parameterize only $f(z)$ — a simpler task.

### Residual block

$$
\text{ResBlock}(z) = \sigma\left(W^{[2]}\, \sigma(W^{[1]} z + b^{[1]}) + b^{[2]}\right) + z
$$

**Key requirement:** input and output must have the **same dimension** (so $z$ can be added back).

**Why 2 layers inside the block?** Empirically, 2 (or 3) inner layers work best. One layer seemed insufficient for modeling the residual.

### ResNet (He et al., 2015)

A **Residual Network** stacks many residual blocks:

```
x → [dim change] → ResBlock → ResBlock → ... → ResBlock → [dim change] → output
```

The dimension change at start/end handles the input/output dimension mismatch.

> 📌 *Lecture:* "You model the differences explicitly because you're already kind of close. The original motivation was: if $z$ is already approximately equal to $y$, you probably should use the network to model the differences. It makes it a simpler task."

### Why residual connections help optimization

Another explanation (supported by theory): the skip connection $+z$ improves the conditioning of the optimization landscape. Without it, deep networks suffer from vanishing/exploding gradients. The skip connection ensures gradients can flow directly from the output back to early layers.

### Residuals are everywhere

Residual connections appear in virtually every modern architecture:
- **ResNet** — image classification
- **Transformer** — every attention block and MLP block has $x + \text{block}(x)$
- **U-Net** — image segmentation skip connections

---

> 🎯 **Interview:** What problem do residual connections solve and how?
> 
> **A:** In deep networks, gradients vanish as they propagate backward through many layers — each multiplication by a weight matrix and activation derivative can shrink them exponentially. Residual connections create a "highway" for gradients: the derivative through $z + f(z)$ w.r.t. $z$ is $I + \frac{\partial f}{\partial z}$. The identity term ensures gradient magnitude is always at least 1, preventing vanishing. This enabled training of networks with 100+ layers (ResNet-152 etc.) that were previously impossible to optimize.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demonstrate gradient flow: plain deep network vs. residual network
np.random.seed(0)

def relu(x): return np.maximum(x, 0)
def drelu(x): return (x > 0).astype(float)

def simulate_gradient_norms(depth, use_residual=False, scale=0.5):
    """Simulate gradient norm at each layer in backward pass."""
    # Initialize weights with small scale
    Ws = [np.random.randn(16, 16) * scale for _ in range(depth)]
    x = np.random.randn(16)

    # Forward pass — collect activations
    zs = [x]
    for l in range(depth):
        z = Ws[l] @ zs[-1]
        if use_residual and zs[-1].shape == z.shape:
            z = relu(z) + zs[-1]
        else:
            z = relu(z)
        zs.append(z)

    # Backward: track gradient norm
    grad = np.ones(16)  # d_loss / d_output
    norms = [np.linalg.norm(grad)]
    for l in reversed(range(depth)):
        drelu_z = drelu(Ws[l] @ zs[l])
        if use_residual:
            grad = Ws[l].T @ (drelu_z * grad) + grad  # skip connection gradient
        else:
            grad = Ws[l].T @ (drelu_z * grad)
        norms.append(np.linalg.norm(grad))
    return list(reversed(norms))

depth = 20
norms_plain = simulate_gradient_norms(depth, use_residual=False, scale=0.4)
norms_resid = simulate_gradient_norms(depth, use_residual=True, scale=0.4)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(norms_plain, 'r-o', ms=5, label='Plain network (no residual)', linewidth=2)
ax.plot(norms_resid, 'b-s', ms=5, label='Residual network', linewidth=2)
ax.set_xlabel('Layer (from input)', fontsize=12)
ax.set_ylabel('Gradient norm', fontsize=12)
ax.set_title(f'Gradient Flow: {depth}-Layer Network — Plain vs Residual', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Plain network — gradient norm at input: {norms_plain[0]:.6f}")
print(f"Residual network — gradient norm at input: {norms_resid[0]:.4f}")
print("\nSkip connection preserves gradient magnitude through depth.")

## 8. Layer Norm and RMSNorm

### The problem: activation explosion

In deep networks, applying many layers can cause activations to grow or shrink exponentially. If activations reach $10^{20}$, everything "blows up" (overflow, NaN). Layer normalization fixes this.

### Layer Norm

Given an input vector $z \in \mathbb{R}^m$, Layer Norm:

**Step 1:** Compute empirical mean and variance:

$$
\hat{\mu} = \frac{1}{m}\sum_{i=1}^m z_i, \qquad \hat{\sigma}^2 = \frac{1}{m}\sum_{i=1}^m (z_i - \hat{\mu})^2
$$

**Step 2:** Normalize to zero mean, unit variance:

$$
\hat{z}_i = \frac{z_i - \hat{\mu}}{\hat{\sigma}}
$$

**Step 3:** Rescale with learnable parameters $\gamma, \beta$:

$$
\text{LayerNorm}(z)_i = \gamma \hat{z}_i + \beta
$$

Why $\gamma, \beta$? Forcing exact mean=0, std=1 might not be optimal. The learnable parameters let the network recover any desired mean/std during training.

### RMSNorm (modern LLMs: LLaMA, Mistral, etc.)

Skip the mean subtraction. Just normalize by the RMS:

$$
\hat{\sigma}_{\text{RMS}} = \sqrt{\frac{1}{m}\sum_{i=1}^m z_i^2}
$$

$$
\text{RMSNorm}(z)_i = \gamma \cdot \frac{z_i}{\hat{\sigma}_{\text{RMS}}}
$$

**Why skip mean subtraction?** Simpler, faster, and empirically works just as well. The mean centering was found to be unnecessary once the network learns to control it via $\gamma$.

### Scale invariance

Key property of Layer Norm:

$$
\text{LayerNorm}(\alpha z) = \text{LayerNorm}(z) \quad \forall \alpha \neq 0
$$

Scaling the input doesn't change the output. This means:
- Weight matrix scaling doesn't change intermediate activations
- Easier initialization (no need to be precise about scale)
- More stable training

> 📌 *Lecture:* "Before, when people tuned all of these models, sometimes you apply these layers many times and the output becomes bigger and bigger — like $10^{20}$ — and then everything just blows up. When you normalize every time, you never get into this issue."

**Subtle note:** the *forward pass* is scale-invariant, but the *backward pass* (gradients) is not. Gradient scaling issues are shifted into the optimizer, not eliminated.

### Where Layer Norm lives in modern architectures

```
Transformer block:
x → LayerNorm → Attention → + x    (residual)
  → LayerNorm → MLP       → + x    (residual)
```

---

> 🎯 **Interview:** What does Layer Norm do and why is it important?
> 
> **A:** Layer Norm normalizes each vector $z$ (a single example's activation at one layer) to zero mean and unit variance, then applies learnable scale $\gamma$ and shift $\beta$. It prevents activations from exploding or vanishing as they pass through many layers, stabilizes training, and reduces sensitivity to weight initialization scale. Unlike Batch Norm (which normalizes across a batch), Layer Norm normalizes each example independently — making it compatible with variable-length sequences, online inference, and small batch sizes, which is why it's used in all major transformer-based models.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def layer_norm(z, gamma=1.0, beta=0.0, eps=1e-8):
    mu = z.mean()
    sigma = z.std() + eps
    return gamma * (z - mu) / sigma + beta

def rms_norm(z, gamma=1.0, eps=1e-8):
    rms = np.sqrt((z**2).mean()) + eps
    return gamma * z / rms

np.random.seed(7)

# Simulate activation growth in a deep network without / with LayerNorm
depth = 30
dim = 64
x = np.random.randn(dim)

norms_no_ln = [np.linalg.norm(x)]
norms_with_ln = [np.linalg.norm(x)]

z_no = x.copy()
z_with = x.copy()

for _ in range(depth):
    W = np.random.randn(dim, dim) * 1.2  # slightly >1 to cause growth
    z_no = np.maximum(W @ z_no, 0)
    norms_no_ln.append(np.linalg.norm(z_no))

    z_with = layer_norm(np.maximum(W @ z_with, 0))
    norms_with_ln.append(np.linalg.norm(z_with))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(norms_no_ln, 'r-o', ms=4, linewidth=2, label='No LayerNorm')
axes[0].plot(norms_with_ln, 'b-s', ms=4, linewidth=2, label='With LayerNorm')
axes[0].set_xlabel('Layer'); axes[0].set_ylabel('||activation||₂')
axes[0].set_title('Activation Norm vs. Depth', fontsize=12)
axes[0].legend(); axes[0].set_yscale('log'); axes[0].grid(True, alpha=0.3)

# Scale invariance demo
z = np.random.randn(16)
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ln_outputs = [layer_norm(alpha * z) for alpha in alphas]
diffs = [np.linalg.norm(out - ln_outputs[2]) for out in ln_outputs]

axes[1].bar(range(len(alphas)), diffs, color='steelblue')
axes[1].set_xticks(range(len(alphas)))
axes[1].set_xticklabels([f'α={a}' for a in alphas])
axes[1].set_ylabel('||LN(αz) - LN(z)||₂')
axes[1].set_title('Scale Invariance: LN(αz) = LN(z)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Layer Norm: Prevents Explosion + Scale Invariant', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Without LN: activation norm after {depth} layers = {norms_no_ln[-1]:.2e}")
print(f"With LN:    activation norm after {depth} layers = {norms_with_ln[-1]:.4f}")
print(f"\nScale invariance: LN(100z) differs from LN(z) by {diffs[-1]:.2e} (essentially 0)")

## 9. Convolutional Networks (CNN)

### Motivation

For a $256 \times 256$ RGB image: $d = 256 \times 256 \times 3 \approx 196$K input dimensions. A fully-connected layer with 1000 hidden units needs $196\text{K} \times 1000 \approx 200$M parameters — per layer. This is wasteful and doesn't use the spatial structure of images.

### The key idea: parameter sharing

A **filter** (kernel) $f \in \mathbb{R}^{k \times k}$ is a small weight matrix. Apply it to every local patch of the image:

$$
(\text{output})_{i,j} = \sum_{p=0}^{k-1} \sum_{q=0}^{k-1} f_{p,q} \cdot x_{i+p, j+q}
$$

The same filter $f$ is applied at **every position** — weights are shared. This is a **convolution**.

### Toeplitz matrix interpretation

In 1D: a convolution with filter $[f_1, f_2, f_3]$ corresponds to multiplying by a Toeplitz matrix:

$$
\begin{pmatrix} f_1 & f_2 & f_3 & 0 & 0 \\ 0 & f_1 & f_2 & f_3 & 0 \\ 0 & 0 & f_1 & f_2 & f_3 \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \\ x_5 \end{pmatrix}
$$

Same parameters $f_1, f_2, f_3$ appear at every row position (shifted). Most entries are zero. A dense layer would have all entries as separate parameters.

### Parameter count

| Layer type | Parameters for $d$-dim input, $m$-dim output |
|---|---|
| Dense (linear) | $m \times d$ |
| Conv (filter size $k$, $m$ filters) | $m \times k$ |

For $d = 196$K and $m = 64$: dense needs 12.5M params; conv with $k=3$ needs 192 params.

### Where CNNs stand today

> 📌 *Lecture:* "Convolutional networks are not used that much except for very hardcore vision cases. Even for vision cases, people use language models like transformers a lot, so it's probably fine to skip them."

CNNs dominated vision from 2012 (AlexNet) to ~2020. Since then, **Vision Transformers (ViT)** have largely replaced them for large-scale vision tasks. CNNs remain competitive for:
- Mobile/edge deployment (efficient architectures: MobileNet, EfficientNet)
- Real-time video
- 3D medical imaging
- Any task where locality/translation invariance is a known inductive bias

---

> 🎯 **Interview:** What is the inductive bias of a CNN and when does it help?
> 
> **A:** CNNs have two inductive biases: (1) **locality** — features are computed from local patches, not the full input; (2) **translation equivariance** — the same filter applied at different positions produces shifted outputs, which is appropriate for images where a cat in the corner and a cat in the center should activate the same detector. These biases are strong priors that help when data is limited. When data is plentiful, transformers (which learn all pairwise relationships via attention) can outperform CNNs by learning the right inductive bias from data rather than baking it in.


## Summary — Neural Network Architecture Toolkit

### The building blocks

```
Input x ∈ ℝᵈ
    │
    ▼
[Linear]  z = Wx + b           ← learned weights W ∈ ℝ^{m×d}, b ∈ ℝ^m
    │
    ▼
[Activation]  a = σ(z)         ← ReLU / GELU / sigmoid (elementwise)
    │
    ├── + skip ──────────────── [Residual connection] (if same dim)
    │
    ▼
[LayerNorm]  normalized a      ← stabilize scale; learnable γ, β
    │
    ▼
[Repeat L times]
    │
    ▼
Output h_θ(x) ∈ ℝᵏ
```

### Key formulas

| Component | Formula | Parameters |
|---|---|---|
| Linear layer | $z = Wx + b$ | $W \in \mathbb{R}^{m \times d}$, $b \in \mathbb{R}^m$ → $m(d+1)$ params |
| ReLU | $\text{ReLU}(t) = \max(t,0)$ | None |
| Residual block | $\text{ResBlock}(z) = \sigma(W_2\sigma(W_1 z+b_1)+b_2) + z$ | Must preserve dim |
| Layer Norm | $\gamma \cdot \frac{z-\mu}{\sigma} + \beta$ | $\gamma, \beta \in \mathbb{R}^m$ |
| RMS Norm | $\gamma \cdot z / \text{RMS}(z)$ | $\gamma \in \mathbb{R}^m$ |

### Two questions answered next lecture

- Q1 (this lecture): **What is $h_\theta$?** → Neural network architecture ✓
- Q2 (L08): **How to compute $\nabla_\theta J$?** → Backpropagation

---

## External Resources

| Resource | What to read | Why |
|---|---|---|
| CS229 Notes Part 5 | §§1–3 (neural networks) | Formal notation + universal approximation |
| Goodfellow *Deep Learning* | Ch. 6 (feedforward networks) | Thorough treatment of MLP |
| He et al. 2016 | "Deep Residual Learning" (ResNet) | Original residual connection paper |
| Ba et al. 2016 | "Layer Normalization" | Original Layer Norm paper |
| Zhang et al. 2019 | "Root Mean Square Layer Normalization" | RMSNorm |
| Dosovitskiy et al. 2020 | "An Image is Worth 16x16 Words" (ViT) | Transformers for vision — why CNNs declined |
| PyTorch docs | `nn.Linear`, `nn.ReLU`, `nn.LayerNorm`, `nn.Conv2d` | Direct implementation reference |
